In [1]:
"""
Week 8 — Evaluation Expansion (MAPE, MdAPE, price-band breakdown)

Deliverable: metrics beyond R2 for the current best model (Gradient
Boosting / XGBoost, light-tuned per Week 7), plus a breakdown of where the
model performs better or worse across price bands.

Design choices:

1. Model: reuses the Week 7 "final, light tuning" search unchanged (grid
   centered on max_depth in [7,9], learning_rate in [0.03,0.05,0.1], with
   n_estimators found via early stopping) so this notebook is self-
   contained — it does not depend on artifacts saved by a previous script.
   The leak-safe machinery (outlier thresholds fit on train only, Sec. 03;
   CV-safe target encoding, Sec. 05; Pipeline/ColumnTransformer fit only on
   train, Sec. 07) is identical to Weeks 4-7.

2. Metrics (Sec. 08): R2 was the only metric reported through Week 7. This
   notebook adds:
     - MAPE (Mean Absolute Percentage Error) — requested explicitly.
     - MdAPE (Median Absolute Percentage Error) — requested explicitly,
       and per Sec. 08 the most representative single number for skewed,
       multi-scale real-estate price data, since a median is barely moved
       by a handful of large-error properties the way MAPE is.
     - MAE is also included (not explicitly requested, but Sec. 08 asks
       for R2/MAPE/MdAPE/MAE to be reported together) so the full
       best-practices metric set is available for the writeup, at
       essentially no extra cost.

3. Price-band breakdown (Sec. 08 "Error Breakdown, Not Just Top-Line
   Numbers"): the test month is split into price quintiles (5 equal-sized
   groups by ClosePrice), and every metric is recomputed within each band.
   Quintile edges come from the TEST month's own ClosePrice distribution
   (this is a post-hoc grouping of already-produced predictions for
   reporting purposes, not a fit-time transform, so it does not raise the
   same leakage concern as, e.g., outlier thresholds or target encoding).
"""

'\nWeek 8 — Evaluation Expansion (MAPE, MdAPE, price-band breakdown)\n\nDeliverable: metrics beyond R2 for the current best model (Gradient\nBoosting / XGBoost, light-tuned per Week 7), plus a breakdown of where the\nmodel performs better or worse across price bands.\n\nDesign choices:\n\n1. Model: reuses the Week 7 "final, light tuning" search unchanged (grid\n   centered on max_depth in [7,9], learning_rate in [0.03,0.05,0.1], with\n   n_estimators found via early stopping) so this notebook is self-\n   contained — it does not depend on artifacts saved by a previous script.\n   The leak-safe machinery (outlier thresholds fit on train only, Sec. 03;\n   CV-safe target encoding, Sec. 05; Pipeline/ColumnTransformer fit only on\n   train, Sec. 07) is identical to Weeks 4-7.\n\n2. Metrics (Sec. 08): R2 was the only metric reported through Week 7. This\n   notebook adds:\n     - MAPE (Mean Absolute Percentage Error) — requested explicitly.\n     - MdAPE (Median Absolute Percentage Error) —

In [2]:
import numpy as np
import pandas as pd
import warnings
from itertools import product

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBRegressor

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = "cleaned_single_family_sales_engineered.csv"   # Week 6 output

TARGET = "ClosePrice"
LOW_CARD_CAT = [
    "PropertySubType", "Levels", "AttachedGarageYN",
    "PoolPrivateYN", "FireplaceYN", "NewConstructionYN",
]
HIGH_CARD_CAT = ["City", "DistrictName"]

BASE_NUMERIC_BASE_NAMES = [
    "LivingArea", "Bedrooms", "Bathrooms", "LotSize",
    "GarageSpaces", "YearBuilt", "ParkingTotal",
]
ENGINEERED_BASE_NAMES = ["BedBathRatio", "PropertyAge"]

FIXED_X = 24   # training-window length (months) — unchanged from Weeks 5-7

In [4]:
df = pd.read_csv(DATA_PATH, parse_dates=["CloseDate"])
print(f"Loaded {DATA_PATH} — shape {df.shape}")

for col in HIGH_CARD_CAT + LOW_CARD_CAT:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

NUMERIC_COLS = BASE_NUMERIC_BASE_NAMES + ENGINEERED_BASE_NAMES
NUMERIC_COLS += [f"{c}_missing" for c in NUMERIC_COLS if f"{c}_missing" in df.columns]
print("Numeric columns:", NUMERIC_COLS)

Loaded cleaned_single_family_sales_engineered.csv — shape (333028, 29)
Numeric columns: ['LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize', 'GarageSpaces', 'YearBuilt', 'ParkingTotal', 'BedBathRatio', 'PropertyAge', 'LivingArea_missing', 'Bathrooms_missing', 'LotSize_missing', 'GarageSpaces_missing', 'YearBuilt_missing', 'ParkingTotal_missing', 'BedBathRatio_missing', 'PropertyAge_missing']


In [5]:
def month_period(series):
    return series.dt.to_period("M")


def get_window(data, end_month_exclusive, n_months):
    start = end_month_exclusive - n_months
    periods = month_period(data["CloseDate"])
    return data[(periods >= start) & (periods < end_month_exclusive)]


def get_month(data, month):
    return data[month_period(data["CloseDate"]) == month]

In [6]:
def kfold_target_encode(train_series, train_target, apply_series_list,
                         n_splits=5, smoothing=20, seed=RANDOM_SEED):
    """CV-safe target encoding (Sec. 05) — unchanged from Weeks 4-7."""
    train_series = train_series.reset_index(drop=True)
    train_target = train_target.reset_index(drop=True)
    global_mean = train_target.mean()

    encoded_train = np.zeros(len(train_series))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, hold_idx in kf.split(train_series):
        fit_vals, fit_target = train_series.iloc[fit_idx], train_target.iloc[fit_idx]
        stats = fit_target.groupby(fit_vals).agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        encoded_train[hold_idx] = train_series.iloc[hold_idx].map(smoothed).fillna(global_mean).to_numpy()

    full_stats = train_target.groupby(train_series).agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
    encoded_apply = [s.map(full_smoothed).fillna(global_mean).to_numpy() for s in apply_series_list]
    return encoded_train, encoded_apply

In [7]:
def compute_metrics(y_true, y_pred):
    """Sec. 08: R2, MAPE, MdAPE, MAE together — no single metric tells the
    whole story. MAPE/MdAPE are computed as percentages (0-100 scale).
    Rows with y_true == 0 would make percentage error undefined; ClosePrice
    is already constrained to be > 0 by the Sec. 02 cleaning step, so no
    additional guard is needed here."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    pct_errors = np.abs((y_true - y_pred) / y_true) * 100
    return {
        "R2": r2_score(y_true, y_pred),
        "MAPE": pct_errors.mean(),
        "MdAPE": np.median(pct_errors),
        "MAE": mean_absolute_error(y_true, y_pred),
        "N": len(y_true),
    }

In [8]:
def build_preprocessor(numeric_final_cols, cat_final_cols):
    """Leak-safe ColumnTransformer (Sec. 07)."""
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_final_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", min_frequency=10)),
        ]), cat_final_cols),
    ])


def build_pipeline(regressor, numeric_final_cols, cat_final_cols):
    preprocessor = build_preprocessor(numeric_final_cols, cat_final_cols)
    return Pipeline([("preprocess", preprocessor), ("regressor", regressor)])

In [9]:
def prepare_train_eval(train_df, eval_df, numeric_cols, high_card_cat):
    """Outlier filtering (Sec. 03, train-only thresholds) and CV-safe
    target encoding (Sec. 05)."""
    train_df = train_df.copy()
    eval_df = eval_df.copy()

    lo, hi = train_df[TARGET].quantile([0.005, 0.995])
    train_df = train_df[(train_df[TARGET] >= lo) & (train_df[TARGET] <= hi)]
    eval_df = eval_df[(eval_df[TARGET] >= lo) & (eval_df[TARGET] <= hi)]

    y_train = train_df[TARGET].reset_index(drop=True)
    y_eval = eval_df[TARGET].reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET, "CloseDate"]).reset_index(drop=True)
    X_eval = eval_df.drop(columns=[TARGET, "CloseDate"]).reset_index(drop=True)

    enc_cols = []
    for col in high_card_cat:
        enc_train, (enc_eval,) = kfold_target_encode(X_train[col], y_train, [X_eval[col]])
        enc_name = f"{col}_target_enc"
        X_train[enc_name] = enc_train
        X_eval[enc_name] = enc_eval
        enc_cols.append(enc_name)

    numeric_final = numeric_cols + enc_cols
    return X_train, y_train, X_eval, y_eval, numeric_final

In [10]:
def fit_and_evaluate(train_df, eval_df, regressor, numeric_cols, low_card_cat, high_card_cat):
    """Fits the leak-safe pipeline on train_df only. Returns the fitted
    model, train/eval metric dicts (now the full R2/MAPE/MdAPE/MAE set),
    and the raw eval actual/predicted arrays for the downstream price-band
    breakdown."""
    X_train, y_train, X_eval, y_eval, numeric_final = prepare_train_eval(
        train_df, eval_df, numeric_cols, high_card_cat
    )
    model = build_pipeline(regressor, numeric_final, low_card_cat)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)
        model.fit(X_train, y_train)
        y_pred_eval = model.predict(X_eval)
        y_pred_train = model.predict(X_train)

    eval_metrics = compute_metrics(y_eval, y_pred_eval)
    train_metrics = compute_metrics(y_train, y_pred_train)
    return model, train_metrics, eval_metrics, X_eval, y_eval, y_pred_eval

In [11]:
def search_with_early_stopping(train_df, eval_df, max_depth, learning_rate,
                                numeric_cols, low_card_cat, high_card_cat,
                                n_estimators_cap=2000, early_stopping_rounds=30):
    """Search-phase fit ONLY (validation month, never the test month) —
    unchanged from the Week 7 final script. Uses plain R2 (via
    r2_score directly) since the search itself doesn't need the fuller
    metric set — that's only computed once, at the final test evaluation.
    """
    X_train, y_train, X_eval, y_eval, numeric_final = prepare_train_eval(
        train_df, eval_df, numeric_cols, high_card_cat
    )

    preprocessor = build_preprocessor(numeric_final, low_card_cat)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)
        X_train_t = preprocessor.fit_transform(X_train, y_train)
        X_eval_t = preprocessor.transform(X_eval)

    regressor = XGBRegressor(
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators_cap,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="rmse",
        early_stopping_rounds=early_stopping_rounds,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    regressor.fit(X_train_t, y_train, eval_set=[(X_eval_t, y_eval)], verbose=False)

    best_iteration = regressor.best_iteration
    y_pred_eval = regressor.predict(X_eval_t, iteration_range=(0, best_iteration + 1))
    val_r2 = r2_score(y_eval, y_pred_eval)
    return best_iteration, val_r2

In [12]:
# ---------------------------------------------------------------------------
# Chronological split (Sec. 01)
# ---------------------------------------------------------------------------
months_sorted = sorted(month_period(df["CloseDate"]).unique())
if len(months_sorted) < 3:
    raise ValueError("Need at least 3 distinct months for a train/validation/test split.")

test_month = months_sorted[-1]
val_month = months_sorted[-2]
print(f"Validation month: {val_month}   Test month: {test_month}")

Validation month: 2026-05   Test month: 2026-06


In [13]:
# ---------------------------------------------------------------------------
# Light, evidence-based hyperparameter grid — identical to the Week 7 final
# script (centered on the interior peak found by that week's wider search).
# ---------------------------------------------------------------------------
PARAM_GRID = {
    "max_depth": [7, 9],
    "learning_rate": [0.03, 0.05, 0.1],
}
N_ESTIMATORS_CAP = 2000
EARLY_STOPPING_ROUNDS = 30

param_combos = [
    dict(zip(PARAM_GRID.keys(), values))
    for values in product(*PARAM_GRID.values())
]

train_window = get_window(df, val_month, FIXED_X)
val_set = get_month(df, val_month)
print(f"Tuning window: {FIXED_X} months before {val_month} "
      f"({len(train_window)} rows) -> validating on {val_month} ({len(val_set)} rows)")

search_rows = []
for params in param_combos:
    best_iteration, val_r2 = search_with_early_stopping(
        train_window, val_set,
        params["max_depth"], params["learning_rate"],
        NUMERIC_COLS, LOW_CARD_CAT, HIGH_CARD_CAT,
        n_estimators_cap=N_ESTIMATORS_CAP,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    search_rows.append({**params, "n_estimators": best_iteration + 1, "Val R2": round(val_r2, 4)})
    print(f"  depth={params['max_depth']:>2} lr={params['learning_rate']:<4} "
          f"-> {best_iteration + 1} trees, val R2={val_r2:.4f}")

search_df = pd.DataFrame(search_rows).sort_values("Val R2", ascending=False).reset_index(drop=True)
best_row = search_df.iloc[0]
best_params = {
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators"]),
}
print(f"\nSelected hyperparameters: {best_params}")

Tuning window: 24 months before 2026-05 (265957 rows) -> validating on 2026-05 (12019 rows)
  depth= 7 lr=0.03 -> 1347 trees, val R2=0.8645
  depth= 7 lr=0.05 -> 660 trees, val R2=0.8623
  depth= 7 lr=0.1  -> 437 trees, val R2=0.8640
  depth= 9 lr=0.03 -> 900 trees, val R2=0.8672
  depth= 9 lr=0.05 -> 622 trees, val R2=0.8678
  depth= 9 lr=0.1  -> 253 trees, val R2=0.8652

Selected hyperparameters: {'max_depth': 9, 'learning_rate': 0.05, 'n_estimators': 622}


In [14]:
# ---------------------------------------------------------------------------
# Final fit: retrain on the window immediately preceding the test month,
# score once on the test month.
# ---------------------------------------------------------------------------
final_train = get_window(df, test_month, FIXED_X)
test_set = get_month(df, test_month)

final_regressor = XGBRegressor(
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
    n_estimators=best_params["n_estimators"],
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

final_model, train_metrics, test_metrics, _X_test, y_test_actual, y_test_pred = fit_and_evaluate(
    final_train, test_set, final_regressor, NUMERIC_COLS, LOW_CARD_CAT, HIGH_CARD_CAT
)

print("=== Overall Test-Month Metrics (Sec. 08: R2, MAPE, MdAPE, MAE) ===\n")
for name, value in test_metrics.items():
    if name == "N":
        print(f"  N:      {value}")
    elif name in ("MAPE", "MdAPE"):
        print(f"  {name}:  {value:.2f}%")
    elif name == "MAE":
        print(f"  {name}:    ${value:,.0f}")
    else:
        print(f"  {name}:    {value:.4f}")

print(f"\nTrain R2: {train_metrics['R2']:.4f}   Test R2: {test_metrics['R2']:.4f}   "
      f"Gap: {train_metrics['R2'] - test_metrics['R2']:.4f}")

=== Overall Test-Month Metrics (Sec. 08: R2, MAPE, MdAPE, MAE) ===

  R2:    0.8703
  MAPE:  15.05%
  MdAPE:  9.86%
  MAE:    $189,284
  N:      12703

Train R2: 0.9438   Test R2: 0.8703   Gap: 0.0735


In [15]:
# ---------------------------------------------------------------------------
# Price-band breakdown (Sec. 08): split the TEST month into price quintiles
# and recompute error metrics within each band. Band edges come from the
# test month's own actual ClosePrice distribution.
#
# R2 is deliberately NOT included per band. R2 measures improvement over
# predicting the group's own mean, and narrowing to a price quintile
# collapses the within-band price variance far more than it shrinks the
# model's absolute dollar errors — so R2 can (and does, in practice) come
# out strongly negative for a perfectly reasonable model once you slice
# into a narrow band. It's a range-dependent metric and becomes unstable
# exactly when the range is restricted, which is why Sec. 08 only asks for
# MAPE/MdAPE broken out by band, not R2.
# ---------------------------------------------------------------------------
N_BANDS = 5
band_labels = [f"Q{i+1}" for i in range(N_BANDS)]
price_bands = pd.qcut(y_test_actual, q=N_BANDS, labels=band_labels)

band_rows = []
for band in band_labels:
    mask = (price_bands == band).to_numpy()
    band_metrics = compute_metrics(y_test_actual[mask], y_test_pred[mask])
    band_rows.append({
        "Segment": f"Price {band}",
        "Price range": f"${y_test_actual[mask].min():,.0f} – ${y_test_actual[mask].max():,.0f}",
        "N": band_metrics["N"],
        "MAPE (%)": round(band_metrics["MAPE"], 2),
        "MdAPE (%)": round(band_metrics["MdAPE"], 2),
        "MAE ($)": round(band_metrics["MAE"], 0),
    })

price_band_df = pd.DataFrame(band_rows)
print("=== Error by Price Band (Q1 = lowest-priced quintile, Q5 = highest) ===\n")
print(price_band_df.to_string(index=False))

=== Error by Price Band (Q1 = lowest-priced quintile, Q5 = highest) ===

 Segment             Price range    N  MAPE (%)  MdAPE (%)  MAE ($)
Price Q1     $190,000 – $575,000 2544     20.35       9.60  80965.0
Price Q2     $575,772 – $800,000 2544     11.91       7.71  82062.0
Price Q3   $801,000 – $1,100,000 2559     12.95       9.02 121869.0
Price Q4 $1,100,005 – $1,650,000 2523     13.59      10.59 184391.0
Price Q5 $1,652,500 – $8,495,000 2533     16.45      13.27 478743.0


In [16]:
# ---------------------------------------------------------------------------
# Auto-generated insight summary — reads directly off the computed table
# above rather than asserting a fixed narrative, since which band performs
# best/worst depends on the actual data and shouldn't be assumed in advance.
# ---------------------------------------------------------------------------
best_band = price_band_df.loc[price_band_df["MdAPE (%)"].idxmin()]
worst_band = price_band_df.loc[price_band_df["MdAPE (%)"].idxmax()]

print("=== Insights ===\n")
print(f"Best-performing band by MdAPE:  {best_band['Segment']} "
      f"({best_band['Price range']}) — MdAPE={best_band['MdAPE (%)']:.2f}%, "
      f"MAPE={best_band['MAPE (%)']:.2f}%")
print(f"Worst-performing band by MdAPE: {worst_band['Segment']} "
      f"({worst_band['Price range']}) — MdAPE={worst_band['MdAPE (%)']:.2f}%, "
      f"MAPE={worst_band['MAPE (%)']:.2f}%")

=== Insights ===

Best-performing band by MdAPE:  Price Q2 ($575,772 – $800,000) — MdAPE=7.71%, MAPE=11.91%
Worst-performing band by MdAPE: Price Q5 ($1,652,500 – $8,495,000) — MdAPE=13.27%, MAPE=16.45%


In [17]:
"""
Documented insights
--------------------
Read alongside the printed price-band table above.

Why MAPE and MdAPE together (Sec. 08):
  - MAPE (mean) is pulled upward by any properties with unusually large
    percentage errors — sparse comps, misclassified subtype, unreported
    renovations. It answers "how bad are the worst misses, on average."
  - MdAPE (median) is barely moved by those same outliers, so it answers a
    different question: "how far off is a typical prediction." For
    right-skewed price data, MdAPE is usually the more representative
    single number, and the gap between MAPE and MdAPE in the price-band
    table is itself informative — a wide gap in a given band means that
    band's errors are unevenly distributed (a few bad misses, not
    uniformly-mediocre predictions), while a narrow gap means errors in
    that band are more evenly spread.

Price-band pattern: see the auto-generated "Insights" printout above for
this run's actual best/worst band — deliberately not hardcoded here, since
which band wins depends on the data and would go stale (or be wrong) if
this notebook is re-run on a new snapshot.

Known limitations / next steps (Sec. 10):
  - Price-band edges are quintiles of the TEST month's own ClosePrice
    distribution, so band boundaries (dollar cutoffs) will shift slightly
    if this notebook is re-run on a different test month — the band
    *labels* (Q1-Q5) are comparable across runs, the dollar ranges are not.
  - Rolling-origin backtesting (Sec. 01) has not been re-run with these
    expanded metrics; all figures above are a single test-month cutoff.
"""

'\nDocumented insights\n--------------------\nRead alongside the printed price-band table above.\n\nWhy MAPE and MdAPE together (Sec. 08):\n  - MAPE (mean) is pulled upward by any properties with unusually large\n    percentage errors — sparse comps, misclassified subtype, unreported\n    renovations. It answers "how bad are the worst misses, on average."\n  - MdAPE (median) is barely moved by those same outliers, so it answers a\n    different question: "how far off is a typical prediction." For\n    right-skewed price data, MdAPE is usually the more representative\n    single number, and the gap between MAPE and MdAPE in the price-band\n    table is itself informative — a wide gap in a given band means that\n    band\'s errors are unevenly distributed (a few bad misses, not\n    uniformly-mediocre predictions), while a narrow gap means errors in\n    that band are more evenly spread.\n\nPrice-band pattern: see the auto-generated "Insights" printout above for\nthis run\'s actual best/

In [18]:
# ---------------------------------------------------------------------------
# Save metrics_summary.csv: an "Overall" row plus one row per price band.
# R2 is kept for the Overall row (it's meaningful across the full test-
# month price range) but is NOT computed per band (see the price-band cell
# above for why) — so R2 will show as blank/NaN on the Price Q1-Q5 rows in
# the saved CSV. That's intentional, not a missing value.
# ---------------------------------------------------------------------------
overall_row = pd.DataFrame([{
    "Segment": "Overall",
    "Price range": f"${y_test_actual.min():,.0f} – ${y_test_actual.max():,.0f}",
    "N": test_metrics["N"],
    "R2": round(test_metrics["R2"], 4),
    "MAPE (%)": round(test_metrics["MAPE"], 2),
    "MdAPE (%)": round(test_metrics["MdAPE"], 2),
    "MAE ($)": round(test_metrics["MAE"], 0),
}])

metrics_summary = pd.concat([overall_row, price_band_df], ignore_index=True)
metrics_summary.to_csv("metrics_summary.csv", index=False)
print("Saved metrics_summary.csv\n")
print(metrics_summary.to_string(index=False))

Saved metrics_summary.csv

 Segment             Price range     N     R2  MAPE (%)  MdAPE (%)  MAE ($)
 Overall   $190,000 – $8,495,000 12703 0.8703     15.05       9.86 189284.0
Price Q1     $190,000 – $575,000  2544    NaN     20.35       9.60  80965.0
Price Q2     $575,772 – $800,000  2544    NaN     11.91       7.71  82062.0
Price Q3   $801,000 – $1,100,000  2559    NaN     12.95       9.02 121869.0
Price Q4 $1,100,005 – $1,650,000  2523    NaN     13.59      10.59 184391.0
Price Q5 $1,652,500 – $8,495,000  2533    NaN     16.45      13.27 478743.0
